# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [1]:
from google.colab import userdata
HF_TOKEN=userdata.get('hf_key')

In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- What one row means for your lane: One row represents one page-query pair for one reporting period (month). Each row summarizes how a specific page performed for a specific search query during that month.

- Which table(s) you'll use: I'll be using the ***fact_content_daily_performance(78.8M)*** and the ***dim_content(520k)*** and ***fact_query_90d*** Tables

- Which time window: I'll pick September, October, and November 2025

- What you'd predict or rank (label or proxy): The model predicts the page's expected click-through rate (CTR). Needed for calculating the opportunity i.e. (Expected_CTR - Actual_CTR)

- One thing you deliberately exclude: I deliberately exclude the click column. the click column with the impression column is used for calculating the cTR. including the click column will leak the CTR and affect the accuracy of our model.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Features: ==>>> [main intent, impression, average_position, search_volume, content_type]

- label: ==>>> [Click_Through_Rate (CTR)] --> clicks/impression

- exclude: ===>>> [Clicks]

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The grain (one row really is what you said),
your slice’s row count and date span,
availability — filter with IS TRUE and show how many rows survive

- The Grain ==>> Checking if a row really contain information about a single content over the last 30days


In [5]:
data = con.sql(f"""SELECT
    COUNT(*) AS rows,
    COUNT(content_hash_id) AS unique_content
FROM {TABLES['fact_daily']}
WHERE report_date > '2025-08-31'::DATE - INTERVAL 60 DAY

""").df()

print(f'{len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 rows


,rows,unique_content
0,77603955,77603955


- Slice's row count -->>

In [8]:
data = con.sql(f"""SELECT
    COUNT(*) AS row_count,
    '2025-08-31'::DATE AS youngest_content,
    '2025-08-31'::DATE - INTERVAL 2 MONTH AS oldest_content
FROM {TABLES['fact_daily']}
WHERE report_date > '2025-08-31'::DATE - INTERVAL 60 DAY
""").df()

print(f'{len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 rows


,row_count,youngest_content,oldest_content
0,77603955,2025-08-31,2025-06-30


- Checking for availability -->

In [9]:
data_aval = con.sql(f"""SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE ga4_data_available IS TRUE
""").df()

print(f'{len(data_aval):,} rows')
data_aval.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 rows


,available_rows
0,2816455


> The feature-vector build (15,310 rows), the feature notes, and the leakage hunt (train-with / train-without + grouped split) now live in `w03_feature_leakage_check.ipynb` (ML-05).

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can identify patterns and associations, but it cannot explain why those patterns occur.

Specifically, this data cannot tell us:

- Whether changing a page title, meta description, or content will cause CTR to increase.
- How Google's ranking or search algorithms make decisions.
- The impact of SERP features (featured snippets, AI Overviews, knowledge panels, ads) unless those features are explicitly captured in the data.
- Whether changes in CTR are due to competitor activity, seasonality, news events, or shifts in user intent.
- User-level behavior, such as why an individual chose to click or not click a search result.
- Long-term trends or seasonal effects, since this analysis is based on a limited historical window.

As a result, the model should be used as a decision-support tool. It can rank pages that appear to underperform relative to similar pages, helping teams prioritize manual review. It cannot prove that optimizing a recommended page will increase clicks or predict how Google will rank pages in the future.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.